# Pipeline Machine Learning End-to-End dengan TensorFlow Extended (TFX) dan Apache Beam

Notebook ini mengimplementasikan Machine Learning Pipeline end-to-end berstandar produksi untuk estimasi harga rumah (*House Price Prediction*) menggunakan **TensorFlow Extended (TFX)** yang diorkestrasi secara otomatis dengan **Apache Beam (`BeamDagRunner`)**.

Pipeline mencakup 10 komponen terintegrasi:
1. **Validasi & Skema Data Otomatis:** Mengidentifikasi statistik dan anomali data.
2. **Preprocessing Konsisten:** Mencegah *train-serving skew* dengan TensorFlow Transform (`TFT`).
3. **Hyperparameter Tuning Otomatis:** Optimasi arsitektur jaringan menggunakan KerasTuner.
4. **Model Training & Evaluation:** Pelatihan Deep Neural Network dan validasi kualitas berbasis ambang batas (*threshold blessing*) menggunakan TFMA.
5. **Model Pusher:** Otomasi ekspor model *blessed* siap deploy ke direktori TensorFlow Serving.

## Inisialisasi dan Konfigurasi Pipeline

In [2]:
import os
import tensorflow as tf
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

In [3]:
USERNAME = 'erlanggajuni45'
PIPELINE_NAME = f'{USERNAME}-pipeline'

DATA_ROOT = os.path.join(PIPELINE_NAME, 'data')
PIPELINE_ROOT = os.path.join(PIPELINE_NAME, 'pipeline_root')
METADATA_DIR = os.path.join(PIPELINE_NAME, 'metadata')
METADATA_PATH = os.path.join(METADATA_DIR, 'metadata.db')
SERVING_MODEL_DIR = os.path.join(PIPELINE_NAME, 'serving_model')

TRANSFORM_MODULE_FILE = 'modules/transform.py'
TUNER_MODULE_FILE = 'modules/tuner.py'
TRAINER_MODULE_FILE = 'modules/trainer.py'

In [4]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(SERVING_MODEL_DIR, exist_ok=True)

## Perancangan Komponen Pipeline End-to-End

Fungsi `init_pipeline` merangkai 10 komponen TFX ke dalam sebuah *Directed Acyclic Graph* (DAG) yang akan dieksekusi oleh Apache Beam:

1. **CsvExampleGen:** Membaca dataset mentah berformat CSV dan membaginya menjadi partisi data latih (*train*) dan evaluasi (*eval*) dalam format standar TFRecord.
2. **StatisticsGen:** Menghitung ringkasan statistik deskriptif dari dataset untuk analisis karakteristik fitur numerik dan kategorikal.
3. **SchemaGen:** Menginferensi skema data secara otomatis, termasuk deteksi tipe data, keberadaan fitur, dan rentang nilai yang diharapkan.
4. **ExampleValidator:** Memvalidasi dataset terhadap skema yang dibuat untuk mendeteksi anomali data, *missing values*, atau ketidaksesuaian tipe fitur.
5. **Transform:** Melakukan rekayasa fitur (*feature engineering*) menggunakan TensorFlow Transform (`modules/transform.py`). Fitur numerik distandarisasi (*z-score*) dan fitur kategorikal dipetakan ke *vocabulary index* untuk menghindari *train-serving skew*.
6. **Tuner (Saran 1):** Mengoptimasi hyperparameter arsitektur model secara otomatis menggunakan KerasTuner (`modules/tuner.py`) dengan strategi *RandomSearch* untuk menemukan kombinasi *units*, *dropout rate*, dan *learning rate* terbaik.
7. **Trainer:** Melatih arsitektur Keras Deep Neural Network (`modules/trainer.py`) menggunakan hyperparameter terbaik hasil dari komponen Tuner serta mengekspor model lengkap beserta *serving signature* (`serve_tf_examples_fn`).
8. **Resolver (LatestBlessedModelStrategy):** Mengidentifikasi dan mengambil model baseline terbaik sebelumnya yang berstatus *blessed* sebagai pembanding performa.
9. **Evaluator:** Mengevaluasi performa model menggunakan TensorFlow Model Analysis (TFMA) pada metrik Mean Absolute Error (MAE) dan Mean Squared Error (MSE), serta menetapkan status *blessing* jika lolos ambang batas validasi (MSE $\le 10^{14}$).
10. **Pusher:** Menerima model yang telah disetujui (*blessed*) oleh Evaluator dan menyimpannya secara otomatis ke direktori deployment (`serving_model_dir`) untuk disajikan oleh TensorFlow Serving.

In [5]:
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen,
    Evaluator,
    ExampleValidator,
    Pusher,
    SchemaGen,
    StatisticsGen,
    Trainer,
    Transform,
    Tuner,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.proto import pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

In [6]:
def init_pipeline(
    pipeline_root: str,
    pipeline_name: str,
    metadata_path: str,
    data_root: str,
    transform_module_file: str,
    tuner_module_file: str,
    trainer_module_file: str,
    serving_model_dir: str,
) -> pipeline.Pipeline:
  """Membangun dan merangkai 10 komponen Machine Learning Pipeline TFX."""

  # 1. Data Ingestion
  example_gen = CsvExampleGen(input_base=data_root)

  # 2. Ringkasan Statistik
  statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])

  # 3. Inferensi Skema
  schema_gen = SchemaGen(
      statistics=statistics_gen.outputs['statistics'], infer_feature_shape=True
  )

  # 4. Validasi Anomali Data
  example_validator = ExampleValidator(
      statistics=statistics_gen.outputs['statistics'],
      schema=schema_gen.outputs['schema'],
  )

  # 5. Rekayasa Fitur (Transform)
  transform = Transform(
      examples=example_gen.outputs['examples'],
      schema=schema_gen.outputs['schema'],
      module_file=transform_module_file,
  )

  # 6. Hyperparameter Tuning (Saran 1)
  tuner = Tuner(
      module_file=tuner_module_file,
      examples=transform.outputs['transformed_examples'],
      transform_graph=transform.outputs['transform_graph'],
      schema=schema_gen.outputs['schema'],
      train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=40),
      eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=20),
  )

  # 7. Pelatihan Model (Trainer)
  trainer = Trainer(
      module_file=trainer_module_file,
      examples=transform.outputs['transformed_examples'],
      transform_graph=transform.outputs['transform_graph'],
      schema=schema_gen.outputs['schema'],
      hyperparameters=tuner.outputs['best_hyperparameters'],
      train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=100),
      eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=50),
  )

  # 8. Resolver Model Baseline
  model_resolver = Resolver(
      strategy_class=LatestBlessedModelStrategy,
      model=Channel(type=Model),
      model_blessing=Channel(type=ModelBlessing),
  ).with_id('latest_blessed_model_resolver')

  # 9. Evaluasi & Validasi Model (Evaluator)
  eval_config = tfma.EvalConfig(
      model_specs=[tfma.ModelSpec(label_key='price')],
      slicing_specs=[tfma.SlicingSpec()],
      metrics_specs=[
          tfma.MetricsSpec(
              metrics=[
                  tfma.MetricConfig(class_name='MeanAbsoluteError'),
                  tfma.MetricConfig(
                      class_name='MeanSquaredError',
                      threshold=tfma.MetricThreshold(
                          value_threshold=tfma.GenericValueThreshold(
                              upper_bound={'value': 1e14}
                          )
                      ),
                  ),
              ]
          )
      ],
  )

  evaluator = Evaluator(
      examples=example_gen.outputs['examples'],
      model=trainer.outputs['model'],
      baseline_model=model_resolver.outputs['model'],
      eval_config=eval_config,
  )

  # 10. Deployment Model (Pusher)
  pusher = Pusher(
      model=trainer.outputs['model'],
      model_blessing=evaluator.outputs['blessing'],
      push_destination=pusher_pb2.PushDestination(
          filesystem=pusher_pb2.PushDestination.Filesystem(
              base_directory=serving_model_dir
          )
      ),
  )

  components = (
      example_gen,
      statistics_gen,
      schema_gen,
      example_validator,
      transform,
      tuner,
      trainer,
      model_resolver,
      evaluator,
      pusher,
  )

  return pipeline.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      components=components,
      metadata_connection_config=metadata.sqlite_metadata_connection_config(
          metadata_path
      ),
      enable_cache=True,
  )

## Eksekusi Otomatis Pipeline dengan Apache Beam Runner

Komponen yang telah dirangkai di atas dieksekusi secara otomatis dan berurutan menggunakan orchestrator `BeamDagRunner`. Seluruh artefak, metadata eksekusi, serta silsilah data (*data lineage*) dicatat secara terpusat pada basis data SQLite metadata store.

In [9]:
pipeline_instance = init_pipeline(
    pipeline_root=PIPELINE_ROOT,
    pipeline_name=PIPELINE_NAME,
    metadata_path=METADATA_PATH,
    data_root=DATA_ROOT,
    transform_module_file=TRANSFORM_MODULE_FILE,
    tuner_module_file=TUNER_MODULE_FILE,
    trainer_module_file=TRAINER_MODULE_FILE,
    serving_model_dir=SERVING_MODEL_DIR,
)

BeamDagRunner().run(pipeline_instance)

Trial 3 Complete [00h 00m 06s]
val_mean_absolute_error: 539994.0625

Best val_mean_absolute_error So Far: 193582.59375
Total elapsed time: 00h 00m 18s
Results summary
Results in erlanggajuni45-pipeline/pipeline_root/Tuner/.system/executor_execution/14/.temp/14/house_price_tuning
Showing 10 best trials
Objective(name="val_mean_absolute_error", direction="min")

Trial 0 summary
Hyperparameters:
units_1: 32
dropout_rate: 0.2
units_2: 32
learning_rate: 0.01
Score: 193582.59375

Trial 2 summary
Hyperparameters:
units_1: 96
dropout_rate: 0.30000000000000004
units_2: 16
learning_rate: 0.001
Score: 539994.0625

Trial 1 summary
Hyperparameters:
units_1: 64
dropout_rate: 0.4
units_2: 48
learning_rate: 0.001
Score: 544059.375


I0913 21:51:57.639649 1417322 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0913 21:51:57.658796 1557636 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(123, generation: 1)
I0913 21:51:57.658913 1557636 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(122, generation: 1)


Processing ./erlanggajuni45-pipeline/pipeline_root/_wheels/tfx_user_code_trainer-0.0+e956c2a135774109f74dd6ecccd1a35e7d002711badecccb7361c87aa92bb269-py3-none-any.whl
Epoch 1/15
100/100 [==============================] - 25s 14ms/step - loss: 647023624192.0000 - mean_absolute_error: 545538.5625 - val_loss: 492947374080.0000 - val_mean_absolute_error: 518186.2812
Epoch 2/15
100/100 [==============================] - 0s 5ms/step - loss: 559672655872.0000 - mean_absolute_error: 335159.6562 - val_loss: 191272419328.0000 - val_mean_absolute_error: 192141.5625
Epoch 3/15
100/100 [==============================] - 0s 4ms/step - loss: 188022489088.0000 - mean_absolute_error: 163522.7500 - val_loss: 166876512256.0000 - val_mean_absolute_error: 152899.2656
Epoch 4/15
100/100 [==============================] - 0s 5ms/step - loss: 280631476224.0000 - mean_absolute_error: 150250.4375 - val_loss: 156382560256.0000 - val_mean_absolute_error: 142113.0469
Epoch 5/15
100/100 [===========================

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: erlanggajuni45-pipeline/pipeline_root/Trainer/model/15/Format-Serving/assets


INFO:tensorflow:Assets written to: erlanggajuni45-pipeline/pipeline_root/Trainer/model/15/Format-Serving/assets


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


## Verifikasi Model Hasil Ekspor Pusher

Setelah seluruh DAG pipeline selesai dieksekusi oleh Apache Beam, komponen Pusher akan mengekspor artefak model ke direktori serving jika model tersebut dinyatakan *Blessed* oleh Evaluator. Kode di bawah memverifikasi ketersediaan folder timestamp model serving hasil ekspor.

In [10]:
pushed_models = sorted(os.listdir(SERVING_MODEL_DIR))
print(f"Model yang berhasil diekspor di {SERVING_MODEL_DIR}:")
for m in pushed_models:
    print(" - Versi Timestamp:", m)

Model yang berhasil diekspor di erlanggajuni45-pipeline/serving_model:
 - Versi Timestamp: 1789208651
 - Versi Timestamp: 1789215343
 - Versi Timestamp: 1789311174
